In [ ]:
%pylab inline
import svgpathtools as spt
import eucare as ec
import networkx as nx

In [ ]:
def kawasaki_sum(v):
    angles = np.abs(np.array([e['in_angle'] for e in v.incoming_iter()]))
    assert len(angles) % 2 == 0
    return np.sum(angles * (-1) ** np.arange(len(angles)))
    
def max_kawasaki_sum(vertices):
    if isinstance(vertices, ec.half.HalfEdgeGraph):
        vertices = [v for v in vertices.vertices if not v.on_border()]
    return np.max([kawasaki_sum(v) for v in vertices])

#max_kawasaki_sum(G)

In [ ]:
render_settings = dict(
    width=1500, height=1500,
    figsize=(7, 7),
    scale='auto',
    render_edges=True,
    render_faces=False,
    render_vertices=False,
    line_width=0.01,
    face_inset=0.000,
    for_cutting=False
)

In [ ]:
filepath = '/home/roman/Documents/Origami/Oripa/tato.svg'
#filepath = '/home/roman/Documents/Origami/Oripa/minotaur.svg'
paths, attributes = spt.svg2paths(filepath)

In [ ]:
#ec.conversions.EHEG_from_edgelist()
#print(attributes)

In [ ]:
from eucare.overlap import MOUNTAIN, VALLEY

points = []
edges = []
for path, attrs in zip(paths, attributes):
    assert len(path) == 1, f'{path}'
    assert isinstance(path[0], spt.path.Line)
    line = path[0]
    start = np.array([line.start.real, line.start.imag], dtype=np.float32)
    end = np.array([line.end.real, line.end.imag], dtype=np.float32)
    #print(start, end)
    crease_type = None
    if 'red' in attrs['style']:
        crease_type = MOUNTAIN
    elif 'blue' in attrs['style']:
        crease_type = VALLEY
    elif 'gray' in attrs['style']:
        continue
    edge_attrs = dict() if crease_type is None else dict(crease_type=crease_type)
    edges.append((len(points), len(points)+1, edge_attrs))
    points.extend([start, end])
points = np.stack(points)

clustering = ec.overlap.group_closeby(points, 1e-2)

first_occurences = np.argmax(clustering[None] == np.arange(np.max(clustering) + 1)[:, None], axis=1)
merged_points = points[first_occurences]

G = nx.Graph()
G.add_edges_from([(tuple(merged_points[clustering[i]]), tuple(merged_points[clustering[j]]), attrs) 
                  for i, j, attrs in edges])

G = ec.conversions.EHEG_from_nx(G)
G.normalize_positions()

In [ ]:
#cc = ec.classifiers.CountingClassifier(ec.classifiers.lambda_classifier(lambda f: f.area()//0.0001)())
cc = ec.classifiers.congruency_classifier()
for f in G.faces:
    f['color_key'] = cc.classify(f)
G.show(**render_settings)
max_kawasaki_sum(G)

In [ ]:
from eucare.overlap import fold_complete
result = fold_complete(G.copy(), overlap_eps=1e-4, area_eps=1e-6)
result['folded_view_top'].show(**render_settings)
result['CP'].show(**render_settings)

In [ ]:
initial_face = None
for f in G.faces:
    pos = np.array([v['pos'] for v in f.vertex_iter()])
    if np.all(np.min(pos, axis=0) <= 0) and np.all(np.max(pos, axis=0) > 0):
        initial_face = f
G.twocolor_faces(initial_face=initial_face)
G.show(**render_settings)
for f in filter(lambda f: f['color_key'], G.faces):
    for e in f.halfedge_iter():
        e['in_angle'] *= -1
G.recompute_positions()
print(max_kawasaki_sum(G))
G.show(**render_settings)

In [ ]:
def get_over_under_pairs(G, two_coloring_key='color_key'):
    # return list of pairs (f1, f2) with f1 over f2
    # G is assumed to be two-colored
    over_under_pairs = []
    for e in G.halfedges:
        crease_type = e.attributes.get('crease_type', None)
        if crease_type in ('mountain', 'valley') and not (e.on_border() or e.rev.on_border()):
            e_above = e if e.face[two_coloring_key] else e.rev
            if crease_type is 'mountain':
                e_above = e_above.rev
            over_under_pairs.append([e_above.face, e_above.rev.face])
    print('number of pairs', len(over_under_pairs))
    return over_under_pairs

over_under_pairs = get_over_under_pairs(G)
G_over = ec.overlap.overlap_graph(G, eps=1e-4)

In [ ]:
G_over.show(**render_settings)

In [ ]:
from collections import defaultdict
area_threshold = 1e-6

cc = ec.classifiers.lambda_classifier(lambda f: f.area() > area_threshold)()
counts = defaultdict(int)
for f in G_over.faces:
    # over = 0
    # under = 0
    # for e in f.halfedge_iter():
    #     print(e.attributes)
    #f['color_key'] = over / (over + under)
    f['color_key'] = cc.classify(f)
    counts[f['color_key']] += 1
print(counts)
print()
G_over.show(**render_settings)

In [ ]:
#central_face = next(iter(f for f in G.faces if f.order() == ec.prototiles.RegularEuclideanTile(n).make_graph(add_positions=True)[0]
#    G = ec.half.EuclideanPositionHEG(other=G)24))
#over_under_pairs = [(e.rev.face, e.rev.nex.rev.face) for e in central_face.halfedge_iter()]

In [ ]:
ec.overlap.find_face_order(G_over, [], ignore_area_threshold=1e-6)  #, over_under_pairs)

#for e in G.halfedges:
#    print(e['original_face_groups'])


TOP = 'top_side'
BOTTOM = 'bottom_side'


def show_folded(G, side=TOP):
    assert side in (TOP, BOTTOM)
    cc = ec.classifiers.CountingClassifier(ec.classifiers.RepresentationClassifier())
    G = G.copy()
    for f in G.faces:
        #f['color_key'] = len(f['original_faces'])
        try:
            f['color_key'] = cc.classify(f['sorted_original_faces'][0 if side is TOP else -1])
        except IndexError:
            f['color_key'] = 1000
        #print(f['color_key'])
    G.show(**render_settings)

    to_delete = [e
                 for e in G.halfedges
                 if not (e.on_border() or e.rev.on_border()) and e.face['color_key'] is e.rev.face['color_key']]

    G.halfedges.difference_update(to_delete)
    G = ec.conversions.EHEG_from_nx(G.to_networkx_undirected(), {v: v['pos'] for v in G.vertices})
#     to_join = []
#     for v in G.vertices:
#         if not v.on_border() and v.order() == 2:
#             to_join.append(v)
#     for v in to_join:
#         G.join_vertex(v)
    G.recompute_lengths_and_angles()
    cc = ec.classifiers.CountingClassifier(ec.classifiers.lambda_classifier(lambda f: f.area()//0.0001)())
    for f in G.faces:
        # over = 0
        # under = 0
        # for e in f.halfedge_iter():
        #     print(e.attributes)
        #f['color_key'] = over / (over + under)
        f['color_key'] = cc.classify(f)
    G.show(**render_settings)

show_folded(G_over, BOTTOM)
show_folded(G_over, TOP)

In [ ]:
#to solve this properly: every triplet gets an area; 

In [ ]:
rho = {frozenset({(16, 17), (43, 45)}), frozenset({(50, 31), (47, 29), (18, 19)}), frozenset({(11, 9), (0, 10)}), frozenset({(6, 42), (43, 45)}), frozenset({(20, 21), (52, 34), (33, 54)}), frozenset({(32, 30), (51, 42), (13, 58), (14, 22)}), frozenset({(51, 42), (55, 45), (1, 7), (14, 22)}), frozenset({(28, 17), (43, 27)}), frozenset({(47, 54), (53, 15), (33, 29), (52, 50), (34, 31), (25, 26), (20, 18), (21, 19)}), frozenset({(8, 12), (48, 44)}), frozenset({(36, 37), (40, 38), (41, 39)}), frozenset({(43, 2), (6, 5)}), frozenset({(30, 0), (11, 13)}), frozenset({(32, 51), (58, 22), (30, 13)}), frozenset({(9, 10)}), frozenset({(1, 56), (57, 55)}), frozenset({(18, 22), (19, 58), (13, 47), (30, 29), (50, 51), (14, 15), (32, 31), (42, 26)}), frozenset({(51, 5), (6, 42), (55, 2), (43, 45)}), frozenset({(13, 58), (57, 0), (1, 7), (11, 56), (32, 30), (51, 42), (55, 45), (14, 22)}), frozenset({(48, 49), (12, 3)}), frozenset({(39, 3), (23, 24), (46, 37), (16, 35), (36, 49), (40, 10), (41, 4), (9, 38)}), frozenset({(1, 7), (14, 22)}), frozenset({(18, 15), (19, 47), (31, 29), (50, 26)}), frozenset({(44, 45), (16, 46), (8, 7), (4, 24)}), frozenset({(8, 7), (4, 24)}), frozenset({(30, 13)}), frozenset({(46, 44), (8, 4)}), frozenset({(49, 10), (16, 46), (9, 3), (4, 24)}), frozenset({(55, 51), (1, 22)}), frozenset({(20, 53), (52, 25), (21, 54), (34, 33)}), frozenset({(11, 7), (0, 45)}), frozenset({(44, 45), (49, 10), (9, 3), (48, 0), (8, 7), (4, 24), (16, 46), (11, 12)}), frozenset({(58, 56), (57, 32)}), frozenset({(16, 17), (46, 28), (27, 44), (43, 45)}), frozenset({(36, 40), (39, 38), (41, 23), (37, 35)}), frozenset({(9, 10), (46, 49), (4, 3)})}

In [ ]:
counts = defaultdict(int)
for groups in rho:
    counts[len(groups)] += 1
counts

In [ ]:
import numpy as np
import eucare as ec
from eucare.example_tilesets import *
from eucare.example_graphs import from_tiles

def fold(G):
    initial_face = None
    for f in G.faces:
        pos = np.array([v['pos'] for v in f.vertex_iter()])
        if np.all(np.min(pos, axis=0) <= 0) and np.all(np.max(pos, axis=0) > 0):
            initial_face = f
    G.twocolor_faces(initial_face=initial_face)
    for f in filter(lambda f: f['color_key'], G.faces):
        for e in f.halfedge_iter():
            e['in_angle'] *= -1
    G.recompute_positions()

G = from_tiles(platonic(4), 4, vertex_based=True)
#G.normalize_positions()
#G.recompute_lengths_and_angles()
G.show()

# from eucare.overlap import fold_complete
# result = fold_complete(G.copy(), overlap_eps=1e-4, area_eps=1e-6)
# result['folded_view_top'].show(**render_settings)
# result['CP'].show(**render_settings)

fold(G)
G_over = ec.overlap.overlap_graph(G, eps=1e-6)
G_over.show()
crease_assignment = ec.overlap.find_face_order(G_over, [], ignore_area_threshold=1e-6)  #, over_under_pairs)

In [ ]:
render_settings = dict(
    width=1500, height=1500,
    figsize=(7, 7),
    scale='auto',
    render_edges=True,
    render_faces=False,
    render_vertices=False,
    line_width=0.1,
    face_inset=0.000,
    for_cutting=False
)

colors = {
    0: (0, 0, 0),
    1: (1, 0, 0),
    -1: (0, 0, 1)
}

fold(G)

G.show(**render_settings)
for e in G.halfedges:
    e['color_key'] = colors[crease_assignment.get(e, 0)]
G.show(**render_settings)

In [ ]:

comparison_func

In [ ]:
list(G_over.faces)[0].attributes

In [ ]:
s = 0
for e in G_over.halfedges:
    s += len(e['original_edges'])
    print(len(e['original_edges']), e)
    for e2 in e['original_edges']:
        pass
        #print(e2)
print(s)
print(len(G.halfedges))

In [ ]:
s = 0
for e in G_over.halfedges:
    groups = e['original_face_groups']
    for g in groups:
        print(len(g))
    print()
    print(len(groups))
    print()
    s += len(e['original_face_groups'])
    
print(s)
print(len(G.halfedges))